In [ ]:
%pip install "numpy==1.23.5" ultralytics --no-cache-dir
import numpy as np

In [ ]:
!pip install -U ultralytics

In [ ]:
pip install ensemble-boxes

In [ ]:
# core
import os
import json
import random
from typing import Iterable, Tuple, Dict, Any
from ensemble_boxes import weighted_boxes_fusion

# numeric & plotting
import matplotlib.pyplot as plt
from tqdm import tqdm

# vision
import cv2 as cv
import torch

# ultralytics YOLO
from ultralytics import YOLO

# Imports

# Main

## Methods

In [ ]:
def apply_clahe_bgr(frame_bgr: np.ndarray) -> np.ndarray:
    hsv = cv.cvtColor(frame_bgr, cv.COLOR_BGR2HSV)
    h, s, v = cv.split(hsv)
    clahe = cv.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    v2 = clahe.apply(v)
    hsv2 = cv.merge([h, s, v2])
    return cv.cvtColor(hsv2, cv.COLOR_HSV2BGR)

def safe_clip_box(x1, y1, x2, y2, W, H):
    xi1 = max(0, min(int(round(x1)), W - 1))
    yi1 = max(0, min(int(round(y1)), H - 1))
    xi2 = max(0, min(int(round(x2)), W - 1))
    yi2 = max(0, min(int(round(y2)), H - 1))
    # ensure proper order
    if xi2 <= xi1 or yi2 <= yi1:
        return None
    return xi1, yi1, xi2, yi2

def l2_normalize(vec: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(vec)
    if n < eps:
        return vec
    return vec / n

def cosine_similarity(a: np.ndarray, b: np.ndarray, eps: float = 1e-12) -> float:
    # expects 1D float32 vectors
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na < eps or nb < eps:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

def image_to_vector(img_bgr: np.ndarray) -> np.ndarray:
    # Convert to float32 and scale to [0,1], flatten
    v = img_bgr.astype(np.float32) / 255.0
    return v.reshape(-1)

In [ ]:
def compute_similarity_scores(
    frame,
    xyxy: np.ndarray,
    confs: np.ndarray,
    ref_images,
    ref_cache: dict,
    W: int,
    H: int,
) -> list:
    """Return list of similarity-weighted scores (same length as xyxy)."""
    scores = []
    for i in range(xyxy.shape[0]):
        x1, y1, x2, y2 = xyxy[i]
        clipped = safe_clip_box(x1, y1, x2, y2, W, H)
        if clipped is None:
            scores.append(-1e9)
            continue
        xi1, yi1, xi2, yi2 = clipped

        crop = frame[yi1:yi2, xi1:xi2]
        if crop.size == 0:
            scores.append(-1e9)
            continue

        base_conf = float(confs[i])
        best_sim = -10.0

        if ref_images:
            ch, cw = crop.shape[:2]
            key = (cw, ch)
            if key not in ref_cache:
                vecs = []
                for ref_path, ref_img in ref_images:
                    ref_resized = cv.resize(
                        ref_img, (cw, ch), interpolation=cv.INTER_LINEAR
                    )
                    ref_vec = l2_normalize(image_to_vector(ref_resized))
                    vecs.append((ref_path, ref_vec))
                ref_cache[key] = vecs

            crop_vec = l2_normalize(image_to_vector(crop))
            for ref_path, ref_vec in ref_cache[key]:
                sim = cosine_similarity(crop_vec, ref_vec)
                if float(sim) > float(best_sim):
                    best_sim = sim

        combined_score = best_sim * base_conf
        scores.append(combined_score)

    return scores, ref_cache


def extract_yolo_boxes_and_confs(results) -> tuple[np.ndarray, np.ndarray]:
    """Extract xyxy and confs from an Ultralytics Results object."""
    xyxy = results.boxes.xyxy.detach().cpu().numpy()
    confs = results.boxes.conf.detach().cpu().numpy()
    return xyxy, confs

## Inference

In [ ]:
MODEL_PATH_1 = '/kaggle/input/yolov11-weights-zalo-ai-challenge-2025/other/default/5/yolov8n.pt'
MODEL_PATH_2 = '/kaggle/input/yolov11-weights-zalo-ai-challenge-2025/other/default/5/best.pt'

TEST_DATA_DIR = '/kaggle/input/zalo-ai-challenge-2025-track-1-dataset/public_test/samples'

OUTPUT_FILE = 'predictions.json'

REF_IMG_DIR = '/kaggle/input/ref-images-zalo-ai-challenge-2025/Ref'

CONFIDENCE_THRESHOLD = 0.25

In [ ]:
def run_inference(
    TTA: bool = True,
    REF_IMG_DIR: str = REF_IMG_DIR,
    TEST_DATA_DIR: str = TEST_DATA_DIR,
    MODEL_PATH_1: str = MODEL_PATH_1,
    MODEL_PATH_2: str = MODEL_PATH_2,
    OUTPUT_FILE: str = OUTPUT_FILE,
    CLAHE: bool = False,
):
    try:
        model_1 = YOLO(MODEL_PATH_1)
        model_2 = YOLO(MODEL_PATH_2)
        print(f"Successfully loaded model")
    except Exception as e:
        print(f"Error: Could not load model")
        print(e)
        return

    # Gather videos
    try:
        video_folders = sorted([f for f in os.listdir(TEST_DATA_DIR) if os.path.isdir(os.path.join(TEST_DATA_DIR, f))])
    except FileNotFoundError:
        print(f"Error: Test data directory not found at: {TEST_DATA_DIR}")
        return
    if not video_folders:
        print(f"Error: No video folders found in {TEST_DATA_DIR}")
        return
    print(f"Found {len(video_folders)} videos to process...")

    # Preload and validate reference images
    ref_images = []
    ref_cache = {}
    for p in os.listdir(REF_IMG_DIR):
        img = cv.imread(os.path.join(REF_IMG_DIR, p))
        if img is None:
            print(f"Warning: cannot read reference image: {p}")
            continue
        ref_images.append((p, img))
    if not ref_images:
        print("Warning: no valid reference images loaded; cosine similarity will be skipped.")

    all_predictions = []

    for video_folder_name in video_folders:
        video_path = os.path.join(TEST_DATA_DIR, video_folder_name, "drone_video.mp4")
        if not os.path.exists(video_path):
            print(f"Warning: 'drone_video.mp4' not found in {video_folder_name}, skipping.")
            continue

        video_bboxes = []

        try:
            cap = cv.VideoCapture(video_path)
            if not cap.isOpened():
                raise RuntimeError(f"Cannot open video: {video_path}...")
            idx = 0
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                if CLAHE:
                    frame = apply_clahe_bgr(frame)
                    
                if (idx == 0):
                    print(f'Frame Type: {type(frame)}')
                    print(f"Frame Shape: {frame.shape}")
                    
                if frame.dtype != np.uint8: frame = (np.clip(frame, 0, 255)).astype(np.uint8)
                    
                # ================= model_1 =================
                reslist_model_1 = model_1.predict(
                    frame,
                    imgsz=640,
                    conf=CONFIDENCE_THRESHOLD,
                    verbose=False,
                    augment=TTA
                )
                
                if not reslist_model_1:
                    idx += 1
                    continue
                res_model_1 = reslist_model_1[0]

                if res_model_1.boxes is None or len(res_model_1.boxes) == 0:
                    idx += 1
                    continue

                H, W = frame.shape[:2]
                xyxy_model_1, confs_model_1 = extract_yolo_boxes_and_confs(res_model_1)

                weighted_scores_YOLO11n, ref_cache = compute_similarity_scores(
                    frame=frame,
                    xyxy=xyxy_model_1,
                    confs=confs_model_1,
                    ref_images=ref_images,
                    ref_cache=ref_cache,
                    W=W,
                    H=H,
                )

                # ================= model_2 =================
                reslist_model_2 = model_2.predict(
                    frame,
                    imgsz=1024,
                    conf=CONFIDENCE_THRESHOLD,
                    verbose=False,
                    augment=TTA
                )
                
                if not reslist_model_2:
                    idx += 1
                    continue
                res_model_2 = reslist_model_2[0]

                if res_model_2.boxes is None or len(res_model_2.boxes) == 0:
                    idx += 1
                    continue
                    
                xyxy_model_2, confs_model_2 = extract_yolo_boxes_and_confs(res_model_2)
                
                weighted_scores_model_2, ref_cache = compute_similarity_scores(
                    frame=frame,
                    xyxy=xyxy_model_2,
                    confs=confs_model_2,
                    ref_images=ref_images,
                    ref_cache=ref_cache,
                    W=W,
                    H=H,
                )

                # ================= WEIGHTED BOXES FUSION =================
                # WBF expects normalized [x1, y1, x2, y2] in [0,1] for each model
                boxes_1 = xyxy_model_1.astype(float).copy()
                boxes_1[:, 0] /= W  # x1
                boxes_1[:, 2] /= W  # x2
                boxes_1[:, 1] /= H  # y1
                boxes_1[:, 3] /= H  # y2
        
                boxes_2 = xyxy_model_2.astype(float).copy()
                boxes_2[:, 0] /= W
                boxes_2[:, 2] /= W
                boxes_2[:, 1] /= H
                boxes_2[:, 3] /= H
        
                # Use raw YOLO confidences as WBF scores
                scores_1 = np.array(weighted_scores_YOLO11n, dtype=float)
                scores_2 = np.array(weighted_scores_model_2, dtype=float)
        
                # Single-class: all labels = 0
                labels_1 = np.zeros_like(scores_1, dtype=int)
                labels_2 = np.zeros_like(scores_2, dtype=int)
        
                boxes_list = [boxes_1.tolist(), boxes_2.tolist()]
                scores_list = [scores_1.tolist(), scores_2.tolist()]
                labels_list = [labels_1.tolist(), labels_2.tolist()]
        
                model_weights = [1.0, 1.0]
        
                fused_boxes_norm, fused_scores, fused_labels = weighted_boxes_fusion(
                    boxes_list,
                    scores_list,
                    labels_list,
                    weights=model_weights,
                    iou_thr=0.55,
                    skip_box_thr=float(CONFIDENCE_THRESHOLD),
                )
        
                if len(fused_boxes_norm) == 0:
                    idx += 1
                    continue
        
                # De-normalize fused boxes back to pixel coords
                fused_boxes = np.array(fused_boxes_norm, dtype=float)
                fused_boxes[:, 0] *= W
                fused_boxes[:, 2] *= W
                fused_boxes[:, 1] *= H
                fused_boxes[:, 3] *= H
                fused_boxes = fused_boxes.astype(int)
        
                fused_scores = np.array(fused_scores, dtype=float)
        
                # Take best fused box for this frame
                best_idx = int(np.argmax(fused_scores))
                x1_fused, y1_fused, x2_fused, y2_fused = fused_boxes[best_idx]
        
                # Now use (x1_fused, y1_fused, x2_fused, y2_fused) as your final box
                bbox_data = {
                    "frame": int(idx),
                    "x1": int(x1_fused),
                    "y1": int(y1_fused),
                    "x2": int(x2_fused),
                    "y2": int(y2_fused),
                }
                
                video_bboxes.append(bbox_data)
                idx += 1
            cap.release()
            
        except Exception as e:
            print(f"Error while processing video {video_path}: {e}")
            continue

        detections_list = []
        if video_bboxes:
            detections_list.append({"bboxes": video_bboxes})
        final_video_obj = {
            "video_id": video_folder_name,
            "detections": detections_list
        }
        all_predictions.append(final_video_obj)

    try:
        print(f"\nSaving all {len(all_predictions)} video predictions to {OUTPUT_FILE}...")
        with open(OUTPUT_FILE, "w") as f:
            json.dump(all_predictions, f, indent=4)
        print("Inference complete.")
    except Exception as e:
        print(f"Error: Could not write output JSON file: {e}")

run_inference()